<a href="https://colab.research.google.com/github/ludanajoshi/BIS-Lab/blob/main/BIS_Week3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import random

# 1. INPUT: Distance Matrix in km (Node 0 = Central Depot, Nodes 1-4 = Customer Stops)
row0 = list((0, 10, 15, 20, 25))
row1 = list((10, 0, 35, 25, 30))
row2 = list((15, 35, 0, 30, 10))
row3 = list((20, 25, 30, 0, 15))
row4 = list((25, 30, 10, 15, 0))

dist_matrix = list((row0, row1, row2, row3, row4))

N = len(dist_matrix)
num_ants = 10
max_iter = 50
alpha = 1.0        # Pheromone importance
beta = 2.0         # Distance importance
rho = 0.5          # Evaporation rate
Q = 100.0          # Pheromone deposit constant

# 2. INIT
pheromone = list((list((1.0 for _ in range(N))) for _ in range(N)))
best_route = None
min_dist = float('inf')

# 3. MAIN LOOP
for iteration in range(max_iter):
    iteration_routes = list()

    for ant in range(num_ants):
        route = list((0,))  # Start at Central Depot (Node 0)

        # Visit all customer stops
        while len(route) < N:
            current = route[-1]
            unvisited = list((node for node in range(N) if node not in route))

            # Calculate selection probabilities: (τ^α) * ((1/Dist)^β)
            weights = list()
            for next_stop in unvisited:
                tau = pheromone[current][next_stop] ** alpha
                eta = (1.0 / dist_matrix[current][next_stop]) ** beta
                weights.append(tau * eta)

            # Pick Next_Stop probabilistically
            chosen = random.choices(unvisited, weights=weights, k=1)
            next_stop = chosen.pop()
            route.append(next_stop)

        # Return to Central Depot (Node 0)
        route.append(0)

        # Calculate Total_Distance(Route)
        dist = sum(dist_matrix[route[i]][route[i + 1]] for i in range(len(route) - 1))
        iteration_routes.append((route, dist))

        # Update Best_Route if a shorter loop is found
        if dist < min_dist:
            min_dist = dist
            best_route = route

    # 4. UPDATE PHEROMONES
    # Evaporate old trails
    for i in range(N):
        for j in range(N):
            pheromone[i][j] *= (1.0 - rho)

    # Deposit (Q / Dist) on all edges in each ant's route
    for route, dist in iteration_routes:
        deposit = Q / dist
        for i in range(len(route) - 1):
            u, v = route[i], route[i + 1]
            pheromone[u][v] += deposit
            pheromone[v][u] += deposit  # Remove this line for one-way roads

# 5. OUTPUT
print("Optimal Delivery Route:", " -> ".join(map(str, best_route)))
print("Minimum Total Distance:", min_dist, "km")


Optimal Delivery Route: 0 -> 2 -> 4 -> 3 -> 1 -> 0
Minimum Total Distance: 75 km
